In [1]:
%load_ext autoreload
%autoreload 2

In [2]:

import json
import pandas as pd
import datetime as dt
from dateutil.relativedelta import relativedelta

from rockyelevate.wrapper import Session as ELV
from rockyelevate.utils import response_to_dataframe as elv_res_2_df

from rockyclickup.wrapper import Session as RCU
from rockyclickup.utils import response_to_dataframe as rcu_res_2_df, datetime_nearest_day, convert_datetime
from rockyclickup.database_interface import get_all_fields
from rockyclickup.models import (
    Client,
    FSA,
    DCA,
    HSA,
    HRA,
    ADO,
    LSA,
    EDU,
    PKG,
    TRN,
    MODEL_LOOKUP
)


In [3]:
elv = ELV("PROD", multithread=True, max_threads=40)
rcu = RCU()

In [4]:
ELV_ACCOUNT_TYPE_MAP = {
    "HCFSA": "FSA",
    "DCAP": "DCA",
    "HSA": "HSA",
    "HRA": "HRA",
    "TRANSIT": "TRN",
    "PARKING": "PKG",
    "LIFESTYLE": "LSA"
}

LIST_ID_MAP = {
    v.category: k
    for k, v
    in MODEL_LOOKUP.items()
}

In [5]:
# collect all Elevate Organizations

try:
    with open("all_orgs_res.json", 'r') as f:
        all_orgs_res = json.load(f)
        
    all_orgs_df = elv_res_2_df(all_orgs_res)
    print(f"{len(all_orgs_df)} organizations found in file")

except FileNotFoundError:
    all_orgs_res = elv.get_organizations(
        types=["SYSTEM", "PARTNER", "DISTRIBUTOR", "COMPANY", "SUBSIDIARY", "SUBGROUP"],
        statuses=["PENDING", "ACTIVE", "TERMINATED", "ACTIVATION_FAILED"],
        # details=['PARTNER_EXTERNAL_IDENTIFER']
    )
    all_orgs_df = elv_res_2_df(all_orgs_res)
    
    with open("all_orgs_res.json", "w") as f:
        json.dump(all_orgs_res, f, indent=4)

    print(f"{len(all_orgs_res)} organizations found from Elevate API")

1123 organizations found in file


In [6]:
# collect all Elevate plans

try:
    with open("all_plans_res.json", "r") as f:
        all_plans_res = json.load(f)
        
    all_plans_df = elv_res_2_df(all_plans_res)
    print(f"{len(all_plans_df)} plans found in file")

except FileNotFoundError:
    all_oids = [o.get("id") for o in all_orgs_res]
    all_plans_res = elv.get_plans_by_org(oids=all_oids, detail=True)
    all_plans_df = elv_res_2_df(all_plans_res)

    with open("all_plans_res.json", "w") as f:
        json.dump(all_plans_res, f, indent=4)

    print(f"{all_plans_res} plans found from Elevate API")

6926 plans found in file


In [7]:
# collect all ClickUp Clients

try:
    with open("all_clients_res.json", "r") as f:
        all_clients_res = json.load(f)
        
    all_clients_df = rcu_res_2_df(all_clients_res)
    print(f"{len(all_clients_df)} clients found in file")

except FileNotFoundError: 
    all_clients_res = rcu.get_full_list(model=Client)
    all_clients_df = rcu_res_2_df(all_clients_res)

    with open("all_clients_res.json", "w") as f:
        json.dump(all_clients_res, f, indent=4)

    print(f"{len(all_clients_df)} clients found from ClickUp API")

Field not in config.db: update_account_managers 702d85f6-7155-447c-9019-206db17ab27c
Field not in config.db: ducks 80f59460-7b7d-4652-9d9c-f1029b146ede
Field not in config.db: data_transmission_details 2bd919bf-6695-4f13-9f80-9e7b077bc83c
Field not in config.db: divisional_invoicing 951e9aa5-c6d9-4ad1-98a9-118b5d478640
Field not in config.db: temp_am_cobra f1c08a6f-3a1c-4392-8967-bbcd9501e1a8
Field not in config.db: temp_account_manager 159d31bb-e617-4d3a-b900-021cc0ad4542
Field not in config.db: data_start 7d44c3ea-fbe2-42d7-a21a-0e8f42bd8f1b
Field not in config.db: temp_flex_divisions 0d2211c9-b8c9-417e-b020-74ca30273509
Field not in config.db: temp_cobra_divisions e1798c36-372a-446b-8612-de839429d43c
Field not in config.db: data_end 3b8c9ecd-1e23-442a-8155-2b15c1559dee
2797 clients found in file


In [8]:
# collect all ClickUp Plans

try:
    with open("all_clickup_plans_res.json", "r") as f:
        all_plan_responses = json.load(f)

    print(f"{len(all_plan_responses)} clickup plans found in file")

except FileNotFoundError:
    all_plan_responses = []

    for plan_model in [FSA, DCA, HSA, HRA, ADO, LSA, EDU, PKG, TRN]:
        list_response = rcu.get_full_list(model=plan_model)
        all_plan_responses.extend(list_response)

    with open("all_clickup_plans_res.json", "w") as f:
        json.dump(all_plan_responses, f, indent=4)

    print(f"{len(all_plan_responses)} plans fetched from ClickUp API")
    
clickup_plan_df = rcu_res_2_df(all_plan_responses)

6253 clickup plans found in file
Field not in config.db: structure 5515a85d-d6fb-45bf-8e40-081e41b5e11b


In [9]:
# add ClickUp Client details to ClickUp Plan rows

client_merge_df = all_clients_df.copy()
cu_plan_merge_df = clickup_plan_df.copy()

relation_fields = [
    'client_fsa',
    'client_hsa',
    'client_dca',
    'client_hra',
    'client_lsa',
    'client_pkg',
    'client_ado',
    'client_edu',
    'client_trn'
]


# replace empty NaNs with empty lists
for col in relation_fields:
    cu_plan_merge_df[col] = cu_plan_merge_df[col].apply(
        lambda x: x if isinstance(x, list) else []
    )

# combine all client ids from relation fields into single column
cu_plan_merge_df['client_id_list'] = cu_plan_merge_df.apply(
    lambda row: list(set(sum([row[field] for field in relation_fields], []))),
    axis=1
)

# check if there are any plans that have more than one client linked
more_than_one = cu_plan_merge_df[cu_plan_merge_df['client_id_list'].apply(lambda x: len(x) > 1)]
if not more_than_one.empty:
    print(f"{len(more_than_one)} plans have more than one client linked!!!")

# extract first id from the list of client ids (these lists should always have either 1 or 0 items)
cu_plan_merge_df["client_id"] = cu_plan_merge_df['client_id_list'].apply(
    lambda x: x[0] if len(x) > 0 else None
)

# rename columns that appear in both dataframes
matching_columns = [c for c in client_merge_df.columns if c in cu_plan_merge_df.columns]
client_rename_map = { c: f"client_{c}" for c in matching_columns }
plan_rename_map = { c: f"plan_{c}" for c in matching_columns }

client_merge_df = client_merge_df.rename(columns={"id": "client_id"})
client_merge_df = client_merge_df.rename(columns=client_rename_map)
cu_plan_merge_df = cu_plan_merge_df.rename(columns=plan_rename_map)

# finally merge them
client_plan_df = pd.merge(left=cu_plan_merge_df, right=client_merge_df, on="client_id", how='left')

# fix clickup dates

for col in ["date_plan_start", "date_plan_end"]:
    client_plan_df[col] = pd.to_datetime(client_plan_df[col])
    client_plan_df[col] = client_plan_df[col].apply(lambda x: datetime_nearest_day(x))



In [10]:
# add Elevate Organization details onto Elevate Plan rows

all_orgs_df = all_orgs_df.rename(columns={
    "id": "organization_id",
    "parent_id": "organization_parent_id"
})

all_plans_df = all_plans_df.rename(columns={
    "parent_id": "plan_parent_id"
})

org_plan_df = pd.merge(left=all_plans_df, right=all_orgs_df, on='organization_id', how='left')


In [11]:
# filter elevate plan df to rows that start ~january 1st 2026

for col in ['plan_year.valid_from', 'plan_year.valid_to']:
    org_plan_df[col] = pd.to_datetime(org_plan_df[col])

january_df = org_plan_df[
    (org_plan_df['plan_year.valid_from'] >= dt.datetime(2025, 12, 31)) &
    (org_plan_df['plan_year.valid_from'] <= dt.datetime(2026, 1, 2))
]

print(f"{len(january_df)} 01/01/2026 plans")
display(january_df['plan_year.valid_from'].value_counts())

# normalize account_type
january_df['normalized_account_type'] = january_df['account_type.account_type'].apply(lambda x: ELV_ACCOUNT_TYPE_MAP.get(x, x))

2019 01/01/2026 plans


plan_year.valid_from
2026-01-01    2019
Name: count, dtype: int64

C:\Users\james.richmond\AppData\Local\Temp\ipykernel_6712\3060842955.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  january_df['normalized_account_type'] = january_df['account_type.account_type'].apply(lambda x: ELV_ACCOUNT_TYPE_MAP.get(x, x))


In [12]:
# check what elevate plans have already been created clickup

rolled_plan_map = {}

for _, row in january_df.iterrows():
    clickup_plan_search = client_plan_df[
        (client_plan_df["date_plan_start"] == row['plan_year.valid_from']) &
        (client_plan_df["date_plan_end"] == row['plan_year.valid_to']) &
        (client_plan_df["rmr_code"] == row['external_identifier']) &
        (client_plan_df["plan_list.name"] == row['normalized_account_type'])
    ]

    if not clickup_plan_search.empty:
        rolled_plan_map[row['id']] = clickup_plan_search['plan_id'].to_list()

if all([len(x) == 1 for  x in rolled_plan_map.values()]):
    rolled_plan_map = {k: v[0] for k, v in rolled_plan_map.items()}

print(f"{len(rolled_plan_map)} elevate plans have already been created on clickup")


48 elevate plans have already been created on clickup


In [13]:
# try to find last years clickup plan card for each row

previous_plan_card_map = {}

for _, row in january_df.iterrows():
    if pd.isna(row['plan_year.valid_from']) or pd.isna(row['plan_year.valid_to']):
        continue
    
    clickup_plan_search = client_plan_df[
        (
            (client_plan_df["date_plan_start"] == row['plan_year.valid_from'] - relativedelta(years=1)) |
            (client_plan_df["date_plan_end"] == row['plan_year.valid_to'] - relativedelta(years=1)) |
            (
                (client_plan_df["date_plan_start"] <= row['plan_year.valid_from'] - relativedelta(years=1) + relativedelta(days=1)) &
                (client_plan_df["date_plan_start"] >= row['plan_year.valid_from'] - relativedelta(years=1) - relativedelta(days=1))
            )
        ) &
        (client_plan_df["rmr_code"] == row['external_identifier']) &
        (client_plan_df["plan_list.name"] == row['normalized_account_type'])
    ]

    if not clickup_plan_search.empty:
        previous_plan_card_map[row['id']] = clickup_plan_search['plan_id'].to_list()


In [14]:
for elv_plan_id, previous_cu_plans in previous_plan_card_map.items():
    if len(previous_cu_plans) > 1:
        print(f"{elv_plan_id}: {previous_cu_plans}")

93261: ['868c5gaxj', '868c5gawx']
93260: ['868c5gaxj', '868c5gawx']
96046: ['868c5gutu', '868c5gun4']
96047: ['868c5gutu', '868c5gun4']
96555: ['868c5fdyy', '868c5fdxz']
96562: ['868c5fdyy', '868c5fdxz']
93235: ['868c5g30g', '868c5g2zf']
93238: ['868c5g30g', '868c5g2zf']
95231: ['868c5g4gh', '868c5g4fh']
95233: ['868c5g4gh', '868c5g4fh']
95234: ['868c5g4gh', '868c5g4fh']
97164: ['868c6q5nd', '868c492a5']
97176: ['868c5ez7w', '8688t8h80']
97177: ['868c5ez66', '8688t8h7a']
97305: ['868bgm0jn', '868b91bw8']
97309: ['868bgm0jn', '868b91bw8']


In [15]:
client_rmrcode_map = { r['rmr_code']: r['id'] for _, r in all_clients_df.iterrows() }
id_lookup_df = all_orgs_df.copy()[['organization_id', 'external_identifier']]
id_lookup_df['client_id'] = all_orgs_df['external_identifier'].apply(lambda x: client_rmrcode_map.get(x))

In [16]:
print(len(id_lookup_df[id_lookup_df['organization_id'].isna()]))
print(len(id_lookup_df[id_lookup_df['external_identifier'].isna()]))
print(len(id_lookup_df[id_lookup_df['client_id'].isna()]))

org_to_client_map = { r['organization_id']: r['client_id'] for _, r in id_lookup_df.iterrows() }

0
0
7


In [17]:
january_df['client_id'] = january_df['organization_id'].apply(lambda x: org_to_client_map.get(x))

C:\Users\james.richmond\AppData\Local\Temp\ipykernel_6712\1659501238.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  january_df['client_id'] = january_df['organization_id'].apply(lambda x: org_to_client_map.get(x))


In [18]:
[c for c in client_plan_df.columns if "list" in c]

['plan_checklists',
 'plan_list.id',
 'plan_list.name',
 'plan_list.access',
 'client_id_list',
 'client_checklists',
 'client_list.id',
 'client_list.name',
 'client_list.access']

In [19]:
# filter out hsa plans for clients that already have an hsa card
orgs_w_hsa = client_plan_df[client_plan_df['plan_list.name'] == "HSA"]['rmr_code'].unique()
print(len(orgs_w_hsa))

no_dup_hsa = january_df[
    ~(
        (january_df['external_identifier'].isin(orgs_w_hsa)) &
        (january_df['account_type.account_type'] == "HSA")
    )
]

print(len(no_dup_hsa))

672
1508


In [20]:
# narrow january df to columns needed to create plan cards

narrow_df = no_dup_hsa.copy()

narrow_df = narrow_df[[
    "id",
    "client_id",
    "plan_code",
    "normalized_account_type",
    "plan_status",
    "organization_id",
    "external_identifier",
    "plan_year.valid_from",
    "plan_year.valid_to",
    "plan_primary_config.is_carded.is_carded",
    "plan_primary_config.max_election_amount_type.max_election_amount_type",
    "plan_primary_config.max_election_amount_type.max_election_amount",
    # "annual_election_auto_adjust",
    "plan_primary_config.min_election_amount_type.min_election_amount",
    "plan_coverage_config.run_out_type.run_out_days_amount",
    "plan_coverage_config.claims_deadline_end_of_coverage_type.claims_deadline_end_of_coverage_days_amount",
    "plan_coverage_config.claims_deadline_end_of_coverage_type.claims_deadline_end_of_coverage_type",
    "plan_coverage_config.grace_period_type.grace_period_days_amount",
    "plan_account_funding_config.is_rollover.is_rollover",
    "plan_account_funding_config.max_rollover_amount.max_rollover_amount",
    "plan_account_funding_config.min_rollover_amount.min_rollover_amount",
    # "plan_account_funding_config.is_auto_enrollment.is_auto_enrollment"
]].rename(columns={
    "id": "elv_id",
    "external_identifier": "rmrcode",
    "plan_code": "elv_plan_code",
    "plan_year.valid_from": "date_plan_start",
    "plan_year.valid_to": "date_plan_end",
    "plan_primary_config.is_carded.is_carded": "card",
    "plan_primary_config.max_election_amount_type.max_election_amount": "annual_election_max",
    "plan_primary_config.min_election_amount_type.min_election_amount": "annual_election_min",
    "plan_coverage_config.run_out_type.run_out_days_amount": "run_out",
    "plan_coverage_config.claims_deadline_end_of_coverage_type.claims_deadline_end_of_coverage_days_amount": "run_out_termed_ee",
    "plan_coverage_config.claims_deadline_end_of_coverage_type.claims_deadline_end_of_coverage_type": "run_out_termed_ee_matches_plan",
    "plan_coverage_config.grace_period_type.grace_period_days_amount": "grace_period",
    "plan_account_funding_config.is_rollover.is_rollover": "rollover",
    "plan_account_funding_config.max_rollover_amount.max_rollover_amount": "rollover_max",
    "plan_account_funding_config.min_rollover_amount.min_rollover_amount": "rollover_min"
})


In [21]:
# format fields for clickup

def get_card_name(row):
    new_name = f"{row['rmrcode']} {row['normalized_account_type']}"

    if row['normalized_account_type'] != "HSA":
        new_name = f"{new_name} {row['date_plan_start'].year}"

narrow_df["name"] = narrow_df.apply(lambda row: f"{row['rmrcode']} {row['normalized_account_type']} {row['date_plan_start'].year}", axis=1)

def get_lpf(row):
    if row['normalized_account_type'] != "FSA":
        return None
    
    if row['elv_id'] not in previous_plan_card_map:
        return False
    
    plan_ids = previous_plan_card_map.get(row['elv_id'])
    if not plan_ids or len(plan_ids) == 0:
        return False
    
    plan_id = plan_ids[0]

    matching_plans = client_plan_df[client_plan_df['plan_id'] == plan_id]

    if not matching_plans.empty:
        return matching_plans.iloc[0]["lpf"]

    return False
narrow_df['lpf'] = narrow_df.apply(get_lpf, axis=1)

def get_account_manager(row):
    # get account manager from previous plan card
    if row['elv_id'] in previous_plan_card_map:
        plan_id = previous_plan_card_map.get(row['elv_id'])[0]
        matching_plans = client_plan_df[client_plan_df['plan_id'] == plan_id]
        return matching_plans.iloc[0]['plan_account_manager']
    
    # if no previous plan card, get account manager from client
    matching_client = all_clients_df[all_clients_df['rmr_code'] == row['rmrcode']]
    if not matching_client.empty:
        return matching_client.iloc[0]['account_manager']
    
    return None

narrow_df['account_manager'] = narrow_df.apply(get_account_manager, axis=1)

def get_admin_start(row):
    # get date admin start from previous plan card
    if row['elv_id'] in previous_plan_card_map:
        plan_id = previous_plan_card_map.get(row['elv_id'])[0]
        matching_plans = client_plan_df[client_plan_df['plan_id'] == plan_id]
        return matching_plans.iloc[0]['plan_date_admin_start']
    
    # if no preivous plan card, get date plan start from client
    matching_client = all_clients_df[all_clients_df['rmr_code'] == row['rmrcode']]
    if not matching_client.empty:
        return matching_client.iloc[0]['date_admin_start']
    
    return None
narrow_df['date_admin_start'] = narrow_df.apply(get_admin_start, axis=1)

def get_auto_post(row):
    # get auto_post_contributions from previous plan card
    if row['elv_id'] in previous_plan_card_map:
        plan_id = previous_plan_card_map.get(row['elv_id'])[0]
        matching_plans = client_plan_df[client_plan_df['plan_id'] == plan_id]
        return matching_plans.iloc[0]['auto_post_contributions']
    
    return None
narrow_df['auto_post_contributions'] = narrow_df.apply(get_auto_post, axis=1)

# def get_rollover_eligibility(row):
#     if row["plan_account_funding_config.is_auto_enrollment.is_auto_enrollment"] == True:
#         return "Must enroll in following year to receive rollover"
#     else:
#         return None
# narrow_df['rollover_elligibility'] = narrow_df.apply(get_rollover_eligibility, axis=1)


narrow_df['annual_election_auto_adjust'] = narrow_df['plan_primary_config.max_election_amount_type.max_election_amount_type'].apply(lambda x: True if x == "IRS_LIMIT" else False)
narrow_df['annual_election_max'] = narrow_df.apply(
    lambda row:
        0.0 if row['plan_primary_config.max_election_amount_type.max_election_amount_type'] == "UNLIMITED"
        else row['annual_election_max'],
    axis=1
)

narrow_df['run_out_termed_ee_matches_plan'] = narrow_df['run_out_termed_ee_matches_plan'].apply(lambda x: x == "PRE_DEFINED_END_DATE")

narrow_df['list_id'] = narrow_df['normalized_account_type'].apply(lambda x: LIST_ID_MAP.get(x.lower()))


# add client_id to all relation fields
plan_types = [m.category for m in MODEL_LOOKUP.values()]
for plan_type in plan_types:
    narrow_df[f"client_{plan_type}"] = narrow_df['client_id']


In [107]:

all_db_fields = get_all_fields()

field_map = {
    f.custom_name: f
    for f in all_db_fields
}


def format_dict(plan_dict: dict):
    new_dict = {}
    print(f"{plan_dict['elv_id']} {plan_dict['account_manager']}")
    for key, value in plan_dict.items():
    
        if (isinstance(value, list) and len(value) == 0) or (not isinstance(value, list) and pd.isna(value)):
            continue

        if key == "list_id":
            new_dict[key] = int(value)
            continue

        if key not in field_map:
            new_dict[key] = value
            continue

        field_type = field_map.get(key).type

        match field_type:
            case "list_relationship":
                if isinstance(value, list):
                    new_dict[key] = value
                else:
                    new_dict[key] = [value]

            case "users":
                if isinstance(value, list):
                    new_dict[key] = value
                else:
                    new_dict[key] = [value]

            case "short_text":
                new_dict[key] = str(value)

            case "number":
                new_dict[key] = int(value)

            case "currency":
                new_dict[key] = float(value)

            case "checkbox": 
                new_dict[key] = bool(value)

            case "date":
                new_dict[key] = convert_datetime(value, correct_tz_offset=True)

            case _:
                print(field_type)
                    
    return new_dict     


formatted_dicts = [
    format_dict(r.to_dict()) for _, r in narrow_df.iterrows()
]
# plan_dicts = [
#     r.to_dict() for _, r in narrow_df.iterrows()
# ]


# for plan_dict in plan_dicts:
#     new_dict = format_dict(plan_dict)



95260 [90071342]
95285 [75432760]
95258 [75432781]
95251 [90071342]
95256 [90071243]
95277 [75432760]
95257 [90071243]
95259 [90071243]
95293 [81535121]
95309 [75432760]
95307 [75432760]
95286 [81535121]
95305 [75432760]
95322 [75432760]
95292 [75432760]
95319 [75432753]
95289 [75432760]
95314 [75432753]
95250 [75432781]
93245 [75432768]
93246 [75432768]
93613 [75432768]
95308 [90071243]
95313 [90071243]
95311 [90071243]
94793 [75432760]
95265 [75432760]
95264 [75432760]
95267 [75432760]
95280 [75432760]
95325 [75432768]
95324 [75432768]
95301 [90071243]
95310 [90071243]
95304 [90071243]
95316 [90071243]
95270 [75432768]
95266 [75432768]
95290 [75432760]
95295 [75432760]
95288 [75432781]
95279 [75432781]
95294 [75432781]
95297 [75432781]
95273 [75432760]
95278 [75432760]
95282 [90071243]
95291 [90071243]
95284 [90071243]
95287 [90071243]
95269 [75432781]
95272 [75432781]
95767 [75432768]
95766 [75432768]
95296 [75432768]
95299 [75432768]
95326 []
95328 []
95327 [75432768]
95329 [754327

In [23]:
formatted_dicts[0]

{'elv_id': '95260',
 'client_id': '86877zmcc',
 'elv_plan_code': 'RMRECRDCA0101202612312026',
 'normalized_account_type': 'DCA',
 'plan_status': 'ACTIVE',
 'organization_id': 8441,
 'rmrcode': 'RMRECR',
 'date_plan_start': 1767225600000,
 'date_plan_end': 1798675200000,
 'card': True,
 'plan_primary_config.max_election_amount_type.max_election_amount_type': 'IRS_LIMIT',
 'annual_election_max': 7500.0,
 'annual_election_min': 0.0,
 'run_out': 90,
 'run_out_termed_ee_matches_plan': True,
 'rollover': False,
 'name': 'RMRECR DCA 2026',
 'account_manager': [90071342],
 'auto_post_contributions': False,
 'annual_election_auto_adjust': True,
 'list_id': 901102729288,
 'client_opportunity': '86877zmcc',
 'client_contact': '86877zmcc',
 'client_brokerage': ['86877zmcc'],
 'client_client': '86877zmcc',
 'client_fsa': ['86877zmcc'],
 'client_dca': ['86877zmcc'],
 'client_hsa': ['86877zmcc'],
 'client_hra': ['86877zmcc'],
 'client_pkg': ['86877zmcc'],
 'client_trn': ['86877zmcc'],
 'client_lsa': 

In [ ]:
create_responses = {}


for plan_dict in formatted_dicts:
    try:
        model = MODEL_LOOKUP.get(plan_dict.get("list_id"))

        narrow_dict = { k:v for k,v in plan_dict.items() if k in dir(model) and pd.notna(v) }

        narrow_dict['list_id'] = int(narrow_dict["list_id"])

        plan_model = model(**narrow_dict)

        create_res = rcu.create(plan_model)

        create_responses[plan_dict.get("elv_id")] = create_res
    except Exception as e:
        print(f"error creating plan card for {plan_dict.get("elv_id")}")


In [ ]:
with open("create_responses.json", "w") as f:
    json.dump(create_responses, f, indent=4)

In [ ]:
formatted_dicts[0]

In [ ]:
for plan_dict in plan_dicts:
    model = MODEL_LOOKUP.get(plan_dict.get("list_id"))
    print(model)

    narrow_dict = { k:v for k,v in plan_dict.items() if k in dir(model) and pd.notna(v) }
    display(narrow_dict)

    plan_model = model(**narrow_dict)

    # for key, value in narrow_dict.items():
    #     print(f"\n{key}:\n\t{value} ({type(value)})\n\t{field_map.get(key).type if key in field_map else ".."}")


    create_res = rcu.create(plan_model)

    break

In [24]:
create_responses = []
with open("create_responses.json", "r") as f:
    create_responses = json.load(f)

In [34]:
missing_responses = {k: v for k, v in create_responses.items() if pd.isna(v) or not v.get("id")}

In [35]:
missing_responses

{'95793': None,
 '95808': None,
 '95807': None,
 '95804': None,
 '96863': None,
 '97043': None,
 '97154': None,
 '97157': None,
 '97158': None,
 '98996': None,
 '98997': None,
 '94431': None}

In [44]:
plans_to_retry = [ask_dict for ask_dict in formatted_dicts if ask_dict['elv_id'] not in create_responses or ask_dict['elv_id'] in missing_responses]

In [46]:
len(plans_to_retry)

83

In [83]:
deactivated_users = [81524762, 81524777, 75432710]

In [87]:
retry_create_responses = {}

for plan_dict in plans_to_retry:
    try:
        model = MODEL_LOOKUP.get(plan_dict.get("list_id"))
        # display(plan_dict)

        narrow_dict = { k:v for k,v in plan_dict.items() if k in dir(model) and ((not isinstance(v, list) and pd.notna(v)) or (isinstance(v, list) and len(v) > 0)) }
        # display(narrow_dict)

        narrow_dict['account_manager'] = [c for c in narrow_dict['account_manager'] if c not in deactivated_users]

        narrow_dict['list_id'] = int(narrow_dict["list_id"])

        plan_model = model(**narrow_dict)

        create_res = rcu.create(plan_model)

        create_responses[plan_dict.get("elv_id")] = create_res
        
    except Exception as e:
        print(f"error creating plan card for {plan_dict.get("elv_id")}:\n{e}")



error creating plan card for 95808:
'account_manager'
error creating plan card for 93359:
'account_manager'
error creating plan card for 97153:
'account_manager'
error creating plan card for 97154:
'account_manager'
Error patching task (868g9m612): All users must have access to task
Error patching task (868g9m636): All users must have access to task
error creating plan card for 97306:
'account_manager'


In [88]:
final_error_ids = [95808, 93359, 97153, 97154, 97306]

In [ ]:
# rows_to_retry = narrow_df[narrow_df['elv_id'].isin([int(r.get("elv_id")) for r in plans_to_retry])]

In [67]:
rows_to_retry['client_dca']

382     86877zjdx
391     86877zjdx
734     86877zwd4
735     86877zwd4
1146    86877zmzn
          ...    
6398    868adw48d
6855    868dzycxh
6856    868dzycxh
6857    868dzycxh
6860    868dzycxh
Name: client_dca, Length: 83, dtype: object

In [90]:
more_rows_to_retry = narrow_df[narrow_df['elv_id'].isin(final_error_ids)]
more_rows_to_retry

,elv_id,client_id,elv_plan_code,normalized_account_type,plan_status,organization_id,rmrcode,date_plan_start,date_plan_end,card,...,client_trn,client_lsa,client_ado,client_edu,client_cobra,client_data card,client_data file,client_fund,client_inv,client_clm
1979,95808,86877zuwt,RMRSTAFSA0101202612312026,FSA,ACTIVE,8760,RMRSTA,2026-01-01,2026-12-31,True,...,86877zuwt,86877zuwt,86877zuwt,86877zuwt,86877zuwt,86877zuwt,86877zuwt,86877zuwt,86877zuwt,86877zuwt
4707,93359,86877zj8d,RMRARPADO0101202612312026,ADOPTION,ACTIVE,10040,RMRARP,2026-01-01,2026-12-31,False,...,86877zj8d,86877zj8d,86877zj8d,86877zj8d,86877zj8d,86877zj8d,86877zj8d,86877zj8d,86877zj8d,86877zj8d
5667,97153,86877zj72,RMRACANFSA0101202612312026,FSA,ACTIVE,10277,RMRACAN,2026-01-01,2026-12-31,True,...,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72
5672,97154,86877zj72,RMRACANDCA0101202612312026,DCA,ACTIVE,10277,RMRACAN,2026-01-01,2026-12-31,True,...,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72
6198,97306,868b91d1v,RMROMSWEL0101202612312026,SPECIALTY,ACTIVE,12392,RMROMS,2026-01-01,2026-12-31,False,...,868b91d1v,868b91d1v,868b91d1v,868b91d1v,868b91d1v,868b91d1v,868b91d1v,868b91d1v,868b91d1v,868b91d1v


In [92]:
more_rows_to_retry

,elv_id,client_id,elv_plan_code,normalized_account_type,plan_status,organization_id,rmrcode,date_plan_start,date_plan_end,card,...,client_trn,client_lsa,client_ado,client_edu,client_cobra,client_data card,client_data file,client_fund,client_inv,client_clm
1979,95808,86877zuwt,RMRSTAFSA0101202612312026,FSA,ACTIVE,8760,RMRSTA,2026-01-01,2026-12-31,True,...,86877zuwt,86877zuwt,86877zuwt,86877zuwt,86877zuwt,86877zuwt,86877zuwt,86877zuwt,86877zuwt,86877zuwt
4707,93359,86877zj8d,RMRARPADO0101202612312026,ADOPTION,ACTIVE,10040,RMRARP,2026-01-01,2026-12-31,False,...,86877zj8d,86877zj8d,86877zj8d,86877zj8d,86877zj8d,86877zj8d,86877zj8d,86877zj8d,86877zj8d,86877zj8d
5667,97153,86877zj72,RMRACANFSA0101202612312026,FSA,ACTIVE,10277,RMRACAN,2026-01-01,2026-12-31,True,...,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72
5672,97154,86877zj72,RMRACANDCA0101202612312026,DCA,ACTIVE,10277,RMRACAN,2026-01-01,2026-12-31,True,...,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72,86877zj72
6198,97306,868b91d1v,RMROMSWEL0101202612312026,SPECIALTY,ACTIVE,12392,RMROMS,2026-01-01,2026-12-31,False,...,868b91d1v,868b91d1v,868b91d1v,868b91d1v,868b91d1v,868b91d1v,868b91d1v,868b91d1v,868b91d1v,868b91d1v


In [108]:

final_formatted_dicts = [
    format_dict(r.to_dict()) for _, r in more_rows_to_retry.iterrows()
]

95808 []
93359 [90071342]
97153 []
97154 []
97306 [75432781]


In [99]:
retry_create_responses = {}

for plan_dict in final_formatted_dicts:
    try:
        model = MODEL_LOOKUP.get(plan_dict.get("list_id"))
        display(plan_dict)
        

        narrow_dict = { k:v for k,v in plan_dict.items() if k in dir(model) and ((not isinstance(v, list) and pd.notna(v)) or (isinstance(v, list) and len(v) > 0)) }
        # display(narrow_dict)

        narrow_dict['account_manager'] = [c for c in narrow_dict['account_manager'] if c not in deactivated_users]

        narrow_dict['list_id'] = int(narrow_dict["list_id"])

        plan_model = model(**narrow_dict)

        create_res = rcu.create(plan_model)

        create_responses[plan_dict.get("elv_id")] = create_res
        
    except Exception as e:
        print(f"error creating plan card for {plan_dict.get("elv_id")}:\n{e}")

    break


{'elv_id': '95808',
 'client_id': '86877zuwt',
 'elv_plan_code': 'RMRSTAFSA0101202612312026',
 'normalized_account_type': 'FSA',
 'plan_status': 'ACTIVE',
 'organization_id': 8760,
 'rmrcode': 'RMRSTA',
 'date_plan_start': 1767225600000,
 'date_plan_end': 1798675200000,
 'card': True,
 'plan_primary_config.max_election_amount_type.max_election_amount_type': 'IRS_LIMIT',
 'annual_election_max': 6800.0,
 'annual_election_min': 0.0,
 'run_out': 90,
 'run_out_termed_ee_matches_plan': True,
 'rollover': True,
 'rollover_max': 680.0,
 'name': 'RMRSTA FSA 2026',
 'lpf': True,
 'date_admin_start': 1577836800000,
 'auto_post_contributions': False,
 'annual_election_auto_adjust': True,
 'list_id': 901102729124,
 'client_opportunity': '86877zuwt',
 'client_contact': '86877zuwt',
 'client_brokerage': ['86877zuwt'],
 'client_client': '86877zuwt',
 'client_fsa': ['86877zuwt'],
 'client_dca': ['86877zuwt'],
 'client_hsa': ['86877zuwt'],
 'client_hra': ['86877zuwt'],
 'client_pkg': ['86877zuwt'],
 'cl

error creating plan card for 95808:
'account_manager'
